# Lab 22: Clustering Economies — Guided Lab
## ECON 3916: Applied Machine Learning for Economists
### Guided Construction Lab | 44 min Core + 55 min AI + 15 min Extension + take-home Challenge

---

**Learning Objectives:**
- Load a saved copy of the World Development Indicators, drop the World Bank's aggregates, and record which values were imputed
- Standardize the indicators with `StandardScaler`, and explain why K-Means needs it
- Fit K-Means with K=4, and see that each centroid is its cluster's mean
- Choose K with the elbow and the silhouette, and draw the clusters on two principal components
- Compare the clusters with the World Bank's income groups
- Write your own prompt for a function that chooses K, and check the AI's numbers against the lab's

**Dataset:** World Bank World Development Indicators (WDI): 10 development indicators for every
economy the World Bank reports on, downloaded with `wbgapi` on 4 October 2026 and saved, so
everyone works on the same numbers. After the regional and income-group aggregates and the
sparsely reported economies are dropped, 209 remain.

**Prerequisites:** the Part 0 sections of Labs 1-5, and the labs before this one. Lab 16's
`StandardScaler` is used again here; the clustering itself is new.

**How this lab works:** Parts 1-5 are GUIDED: run each cell as it is, then answer the questions.
The AI-Assisted Expansion is required: as in Labs 16 and 19, you write the prompt yourself. Part 6
is an optional extension for two people, and the Challenge is a take-home.

**Plan on about 1 hour 55 minutes** for the required work: setup, Parts 1-5, the AI-Assisted
Expansion, and the README and submission.

**AI in the manual parts:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion while you work on those parts. AI-assisted coding is allowed only where this lab says so.

---

> ### What you write, and what you run
>
> **You write:**
> - The answers to the interpretation questions, in text cells: one each after Parts 3, 4 and 5.
> - **The AI-Assisted Expansion:** your own prompt, from scratch, asking AI for a function that chooses the number of clusters by silhouette and gives back its scores as a table; a revised prompt for round 2 if round 1 shows a `NO`, a missing line or an error (and round 3 if you need it); a short checking log, with the key lines explained in your own words; and two answers. Your prompts, the log and the answers are what is graded, not the AI's code.
>
> **You run and read:** everything else. Setup and Parts 1-5 are working code to learn from. Read each cell before you run it, and make sure you can say what each line does. The check cells in the AI section: run them; you need not follow their code. The AI's function is pasted, not typed.
>
> **Part 6** (optional, with a partner) is one guided cell in which you change one number. **The Challenge** (take-home) has no starter code: you write it in new cells, from the task. The **README** is text only: the prompt at the end is for documentation, not code.

## Setup

Colab has scikit-learn and matplotlib already. The first line installs `wbgapi`, the World Bank's
own package for its data API; only the optional live download in Part 1 uses it. The `!` runs
that line as a terminal command, not as Python.

The lab reads a copy of the World Bank's data saved on 4 October 2026, so your numbers match the
expected outputs below exactly. That matters here: K-Means with one start is sensitive enough that
one country's updated Gini index can reshuffle the clusters.

In [ ]:
# GUIDED — run as-is
# Setup: install the World Bank's data package (for the optional live download), then the imports
!pip install wbgapi scikit-learn matplotlib -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

print('Setup complete.')

## Part 1: Load World Bank WDI Data (GUIDED — 11 min)

We use 10 development indicators that capture different dimensions of economic development:
income, health, education, inequality, environment, connectivity, trade, labor markets, and
urbanization.

They were downloaded from the World Bank's data API with `wbgapi` on 4 October 2026 and saved as
one CSV file, which Step 1 reads. The file has one row per economy, indexed by its three-letter
code (`USA`, `NGA`), with its name (`Country`), the ten indicators, and two columns from the World
Bank's list of economies: `aggregate` (True for a region or group, not a country) and
`income_level` (its income group, used in Part 5).

Each indicator is the economy's most recent **non-empty** value (`wbgapi`'s `mrnev=1`), so two
numbers in one row can come from different years, some decades apart: one internet share of 0% is
from 1990 (American Samoa), one trade share from 1960 (Nigeria), one Gini index from 1992
(Trinidad and Tobago).

- `pd.read_csv(..., index_col='economy')` uses the code column as the row labels;
  `float_precision='round_trip'` reads every number back exactly as it was saved.
- `try` runs the lines under it; if they raise an error (no internet, say), `except` runs instead
  and reads a copy of the file saved beside the notebook.

In [ ]:
# GUIDED — run as-is
# Step 1: load the saved World Bank data, 10 development indicators for every economy
DATA_URL = ('https://raw.githubusercontent.com/RyanPiao/econ3916-student-notebooks/main/'
            '3916/data/lab-ch22-wdi-2026-10.csv')
LOCAL_COPY = 'data/lab-ch22-wdi-2026-10.csv'

try:
    df = pd.read_csv(DATA_URL, index_col='economy', float_precision='round_trip')
except Exception:
    df = pd.read_csv(LOCAL_COPY, index_col='economy', float_precision='round_trip')

indicators = {                                    # World Bank series code: column name
    'NY.GDP.PCAP.PP.CD': 'gdp_per_capita_ppp',
    'SP.DYN.LE00.IN': 'life_expectancy',
    'SP.DYN.IMRT.IN': 'infant_mortality',
    'SE.PRM.ENRR': 'primary_enrollment',
    'SI.POV.GINI': 'gini_index',
    'EN.GHG.CO2.PC.CE.AR5': 'co2_per_capita',     # CO2 per person, excluding land use
    'IT.NET.USER.ZS': 'internet_users_pct',
    'NE.TRD.GNFS.ZS': 'trade_pct_gdp',
    'SL.UEM.TOTL.ZS': 'unemployment_rate',
    'SP.URB.TOTL.IN.ZS': 'urban_population_pct',
}
feature_names = list(indicators.values())
print(f'Rows in the file: {len(df)}')

**Expected output:** `Rows in the file: 264`.

> **Live data (optional, not for the submitted notebook).** To cluster today's numbers instead, in
> a copy of this notebook, replace Step 1's `try`/`except` with:
> ```python
> import wbgapi as wb
> econ = {e['id']: e for e in wb.economy.list()}
> df = wb.data.DataFrame(list(indicators), mrnev=1, labels=True).rename(columns=indicators)
> df['aggregate'] = [bool(econ[c]['aggregate']) for c in df.index]
> df['income_level'] = [econ[c]['incomeLevel'] for c in df.index]
> ```
> Every number from here on will differ from the expected outputs, and the clusters may not be the
> ones the questions describe.

Not all 264 rows are countries. 47 are World Bank **aggregates**: `World`, `Euro area`,
`East Asia & Pacific`. Left in, a region would be clustered next to the countries that make it
up: the same people counted twice. Step 1, continued, removes them, then drops the economies
missing more than four of the ten indicators.

- `df['aggregate']` is True for an aggregate, and `~` flips True and False, so
  `df[~df['aggregate']]` keeps the rest.
- `.dropna(thresh=6, subset=feature_names)` keeps a row only if at least 6 of the 10 indicators
  are present.

In [ ]:
# GUIDED — run as-is
# Step 1, continued: keep real countries, with at least 6 of the 10 indicators
n_before = len(df)
df = df[~df['aggregate']]
print(f'World Bank aggregates removed: {n_before - len(df)}')

df = df.dropna(thresh=6, subset=feature_names)
print(f'Countries with at least 6 of 10 indicators: {len(df)}')

**Expected output:** 47 aggregates removed, and 209 countries left. (Not all are sovereign
states: Hong Kong, Greenland and West Bank and Gaza are among them, as the World Bank reports on
them separately.)

K-Means cannot use a row with a blank in it, so Step 1, continued, fills each blank with its
column's median. First it measures what is missing: `.isna()` is True for a blank, and the mean of
True/False values is the share that are True. Then it records which cells it fills, in ten extra
columns ending `_imputed`. Those columns are documentation, not evidence: they never go into the
clustering. `.fillna(df[feature_names].median())` fills each column with that column's median.
`.idxmax()` gives the name of the column with the largest share missing.

In [ ]:
# GUIDED — run as-is
# Step 1, continued: what is missing, a record of it, then each blank filled with its column's median
missing_pct = df[feature_names].isna().mean() * 100
print('Missing values per indicator, before imputation (%):')
print(missing_pct.round(1).sort_values(ascending=False).to_string())

for f in feature_names:
    df[f + '_imputed'] = df[f].isna()

df[feature_names] = df[feature_names].fillna(df[feature_names].median())

worst = missing_pct.idxmax()
print(f'\n{missing_pct[worst]:.1f}% of {worst} values are now the column median, not a measurement.')

**Expected output:** `gini_index` 18.2%, `unemployment_rate` 11.0%, `trade_pct_gdp` 7.7%,
`infant_mortality` 6.7%, `co2_per_capita` 3.8%, `gdp_per_capita_ppp` 3.3%,
`primary_enrollment` 1.9%, and 0.0% for the other three. Then: 18.2% of `gini_index` values are now
the column median.

Median imputation pulls those countries toward the middle of the data, so they look more average
than they are. A cluster claim that leans on inequality is weaker than the numbers make it appear:
Step 3 counts the filled-in Gini values in each cluster, and they are not spread evenly.

## Part 2: Standardize Features (GUIDED — 5 min)

K-Means measures how far apart two countries are with the ordinary (Euclidean) distance across all
ten indicators. Step 2 shows each indicator's own scale: its mean, standard deviation, minimum and
maximum. `.describe()` computes those and more; `.T` turns the table on its side, one row per
indicator, and `[['mean', 'std', 'min', 'max']]` keeps four of its columns.

In [ ]:
# GUIDED — run as-is
# Step 2: the ten indicators on their own scales
print(df[feature_names].describe().T[['mean', 'std', 'min', 'max']].round(1))

**Expected output:** GDP per capita runs from 1,154.9 to 163,354.0 dollars (PPP), with a standard
deviation of 32,112.8; infant mortality's standard deviation is 16.4, and the Gini index's is 6.4.

Without standardization, one dollar of GDP per capita counts as much as one infant death per 1,000
births. GDP's standard deviation is about 2,000 times infant mortality's, so a typical gap in GDP
swamps a typical gap in any other indicator: the distances, and so the clusters, would be decided
almost entirely by GDP. `StandardScaler` transforms each feature to have mean 0 and standard
deviation 1:

$$z_{ij} = \frac{x_{ij} - \bar{x}_j}{s_j}$$

After standardization, a 1-unit difference in any feature is one standard deviation of that
feature, so each contributes on the same scale. `fit_transform` does two things: *fit* learns each
column's mean and standard deviation, and *transform* applies them. Here it is fitted on all 209
countries: clustering has no test set. In the check below it, `axis=0` takes the mean (or standard
deviation) down each column, one number per indicator.

In [ ]:
# GUIDED — run as-is
# Step 2, continued: every indicator rescaled to mean 0 and standard deviation 1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_names])

print(f'Mean of each feature: {X_scaled.mean(axis=0).round(4)}')
print(f'Std of each feature:  {X_scaled.std(axis=0).round(4)}')
print(f'Shape of feature matrix: {X_scaled.shape}')

**Expected output:** ten means of `0.` (a `-0.` is a tiny negative number rounded to zero), ten
standard deviations of `1.`, and a shape of `(209, 10)`: 209 countries by 10 standardized
features. `X_scaled` is a NumPy array: the numbers only, without the column names.

## Part 3: Fit K-Means (K=4) and Visualize with PCA (GUIDED — 11 min)

Step 3 fits K-Means with K=4 clusters (matching the World Bank's four income groups: Low,
Lower-Middle, Upper-Middle, High). K-Means places 4 centroids, assigns each country to its nearest
centroid, moves each centroid to the mean of its countries, and repeats until the centroids stop moving.

- `init='k-means++'` chooses starting centroids that are spread apart.
- `n_init` is how many times K-Means starts again from new starting centroids, keeping the best
  run: the one with the lowest WCSS, the sum of squared distances from each country to its
  centroid. With `'k-means++'`, `'auto'` means one start.
- `random_state=42` fixes the random choice of starting centroids, so the clusters are the same
  every time you run it. With one start, another seed gives different clusters.
- `.fit_predict(X_scaled)` fits the model and gives back each row's cluster number, 0 to 3. The
  numbers are names only: cluster 3 is not "more" than cluster 0.

In [ ]:
# GUIDED — run as-is
# Step 3: K-Means with K = 4, on the standardized indicators
kmeans = KMeans(n_clusters=4, init='k-means++', n_init='auto', random_state=42)
df['cluster'] = kmeans.fit_predict(X_scaled)

print(df['cluster'].value_counts().sort_index())

**Expected output:** clusters 0 to 3 hold 53, 11, 51 and 94 countries. The sizes are very unequal:
K-Means finds groups, not equal shares.

Step 3, continued, describes each cluster by its averages. `df.groupby('cluster')[feature_names].mean()`
gives one row per cluster and the mean of each indicator among its countries, in the original
units; `.T` puts the indicators down the side. `kmeans.cluster_centers_` holds the centroids in
standardized units, and `scaler.inverse_transform` turns them back into the original units. The
next line prints the largest gap between a centroid and its cluster's mean. The last print counts,
for each cluster, the Gini values Step 1 filled with the median.

In [ ]:
# GUIDED — run as-is
# Step 3, continued: each cluster's mean, and the centroids in the original units
cluster_means = df.groupby('cluster')[feature_names].mean()
print(cluster_means.T.round(1))

centroids = scaler.inverse_transform(kmeans.cluster_centers_)
gap = np.abs(centroids - cluster_means.to_numpy()).max()    # .to_numpy(): the table's numbers as an array
print(f'\nLargest gap between a centroid and its cluster mean: {gap:.6f}')

# the sum of True/False values counts the Trues: filled-in Gini values per cluster
print('\nGini values filled with the median, per cluster:')
print(df.groupby('cluster')['gini_index_imputed'].sum())

**Expected output:** among other rows, average GDP per capita of 5,685.9, 10,964.2, 76,565.5 and
25,959.5 for clusters 0 to 3; life expectancy 65.6, 65.9, 81.3 and 75.1; infant mortality 38.6,
34.1, 5.0 and 12.0. The 11-country cluster stands out on unemployment (22.4%) and the Gini index
(47.7). Then a largest gap of `0.000000`, and filled-in Gini values of 5, 0, 17 and 16
for clusters 0 to 3: a third of the high-income cluster's Gini values are the median.

That last line is the idea of the whole method: a centroid *is* the mean of its countries, on all
ten indicators at once. K-Means summarises each group by an average, so it inherits the average's
weakness: one extreme member pulls it.

Step 3b draws the clusters. Ten dimensions cannot be plotted, so **PCA** finds the two directions
through the standardized data along which the countries are most spread out, and gives each
country a position on them. `PCA(n_components=2)` asks for two; `.fit_transform` gives back a
209 × 2 array; `.explained_variance_ratio_` is each direction's share of the total variance. In the
prints, `:.1%` shows a share as a percentage with one decimal. The chart draws one scatter per
cluster, so each gets its own colour and legend entry: `X_pca[mask, 0]` takes column 0 of the rows
where `mask` is True. PCA is only for the picture: K-Means used all ten indicators.

In [ ]:
# GUIDED — run as-is
# Step 3b: PCA projection to 2D, coloured by cluster
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f'Variance explained by PC1: {pca.explained_variance_ratio_[0]:.1%}')
print(f'Variance explained by PC2: {pca.explained_variance_ratio_[1]:.1%}')
print(f'Total variance explained:  {sum(pca.explained_variance_ratio_):.1%}')

fig, ax = plt.subplots(figsize=(9, 6))
for k in range(4):
    mask = df['cluster'] == k
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], label=f'Cluster {k}', alpha=0.7)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} of the variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} of the variance)')
ax.set_title('K-Means clusters (K=4), drawn on the first two principal components')
ax.legend()
plt.tight_layout()
plt.show()

**Expected output:** PC1 41.9%, PC2 12.9%, 54.8% in total, and a scatter plot of four colours.

The two axes keep 54.8% of the spread and leave out the rest, so two countries that look close on
the chart can be far apart in the ten dimensions K-Means used.

### Interpretation Questions

1. The smallest cluster has 11 countries, the largest 94. Each centroid is the mean of its members. Why would one extreme country move the 11-country cluster's centroid more than it would move the 94-country cluster's, and what does that mean for how far you trust a small cluster's profile?

*Your answer here:*

## Part 4: Elbow Plot + Silhouette Analysis (GUIDED — 9 min)

How do we know K=4 is the right number of clusters? Step 4 fits K-Means again for every K from 2
to 10, with Step 3's settings, and records two measures for each.

1. **WCSS**, the within-cluster sum of squares, `km.inertia_`: each country's squared distance to
   its own centroid, added up. It always falls as K grows, so the **elbow method** looks for the K
   after which it falls more slowly.
2. **The silhouette.** For one country, *a* is its average distance to the other countries in its
   own cluster and *b* its average distance to the countries of the nearest other cluster; its
   silhouette is (*b* − *a*) / max(*a*, *b*), from −1 to 1. Near 1, it sits deep inside its
   cluster; near 0, on a border. `silhouette_score(X, labels)` averages it over every country.
   Higher is better.

`.append` adds one number to the end of a list. `zip(K_range, wcss_list, sil_list)` walks through
the three side by side, one K at a time. In the print, `{k:<3}` left-aligns K in 3 characters and
`{w:>10.1f}` right-aligns the WCSS in 10, with one decimal, so the columns line up. After the loop,
`np.argmax(sil_list)` is the position of the highest silhouette, and `list(K_range)[...]` turns
that position into a K.

In [ ]:
# GUIDED — run as-is
# Step 4: WCSS and silhouette for K = 2 to 10
K_range = range(2, 11)
wcss_list = []
sil_list = []
for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init='auto', random_state=42)
    labels = km.fit_predict(X_scaled)
    wcss_list.append(km.inertia_)
    sil_list.append(silhouette_score(X_scaled, labels))     # on the same standardized data

print('K  | WCSS       | Silhouette')
print('---|------------|-----------')
for k, w, s in zip(K_range, wcss_list, sil_list):
    print(f'{k:<3}| {w:>10.1f} | {s:.4f}')

best_k = list(K_range)[np.argmax(sil_list)]
print(f'\nBest K by silhouette score: {best_k}')

**Expected output:**

| K | WCSS | Silhouette |
|---|---|---|
| 2 | 1488.3 | 0.2729 |
| 3 | 1293.2 | 0.1895 |
| 4 | 1152.5 | 0.2065 |
| 5 | 1078.8 | 0.1589 |
| 6 | 1014.6 | 0.1646 |
| 7 | 920.8 | 0.1640 |
| 8 | 852.3 | 0.1765 |
| 9 | 799.2 | 0.1668 |
| 10 | 732.1 | 0.1792 |

and `Best K by silhouette score: 2`. The AI-Assisted Expansion asks you to reproduce the silhouette
column and this choice with a function written by AI.

Step 4, continued, draws both columns. `'o-'` draws dots joined by lines. The elbow chart has no
sharp elbow: WCSS falls smoothly from 1488 to 732, its biggest drop, 13%, from K=2 to 3, and it
flattens a little after K=4. The silhouette often peaks at K=2 when the data lies mostly along one
long direction, as here (PC1 holds 41.9% of the variance): the best two-way cut is richer against
poorer. At K=2 that is 86 countries averaging about $8,800 of GDP per capita against 123
averaging about $48,900.

In [ ]:
# GUIDED — run as-is
# Step 4, continued: the elbow chart and the silhouette chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(K_range, wcss_list, 'o-')
axes[0].set_xlabel('Number of clusters (K)')
axes[0].set_ylabel('Within-cluster sum of squares (WCSS)')
axes[0].set_title('Elbow method')

axes[1].plot(K_range, sil_list, 'o-', color='green')
axes[1].set_xlabel('Number of clusters (K)')
axes[1].set_ylabel('Mean silhouette score')
axes[1].set_title('Silhouette analysis')

plt.tight_layout()
plt.show()

### Interpretation Questions

1. Which K would you report for these countries, the silhouette's 2 or Step 3's 4, and why? What does the highest silhouette in the table, 0.27, far below 1, say about how separate the groups are?

*Your answer here:*

## Part 5: Compare Clusters to World Bank Income Groups (GUIDED — 8 min)

The World Bank classifies economies into four income groups by one number, GNI per capita, with
fixed thresholds:
- **L** = Low income
- **LM** = Lower-middle income
- **UM** = Upper-middle income
- **H** = High income

Each group is set by where a country's GNI per capita falls against those fixed lines, one
number; each K-Means cluster is set by distance across all ten indicators. How far do the two
agree? Step 5 reads each country's group from the file's `income_level` column, which holds a code
such as `'HIC'`. `.map(dictionary)` replaces each code by what the dictionary gives for it, and a
code the dictionary lacks becomes blank, so `.fillna('Unknown')` labels it. `pd.crosstab(rows, columns,
margins=True)` counts the countries in each pair of cluster and group, with `All` totals.

In [ ]:
# GUIDED — run as-is
# Step 5: cross-tabulate clusters against World Bank income groups
income_map = {'LIC': 'Low', 'LMC': 'Lower-Middle', 'UMC': 'Upper-Middle', 'HIC': 'High'}
df['wb_income'] = df['income_level'].map(income_map).fillna('Unknown')

df_compare = df[df['wb_income'] != 'Unknown']      # an economy with no group has nothing to compare
ct = pd.crosstab(df_compare['cluster'], df_compare['wb_income'], margins=True)
print(ct)

**Expected output:**

```
wb_income  High  Low  Lower-Middle  Upper-Middle  All
cluster
0             1   24            23             5   53
1             0    0             8             3   11
2            50    0             0             1   51
3            27    1            16            50   94
All          78   25            47            59  209
```

Every one of the 209 countries has an income group, so none is dropped. Perfect agreement would put
each cluster's countries in one column.

### Interpretation Questions

1. The 94-country middle-income cluster mixes 27 High-income economies with 50 Upper-Middle ones. What kind of economy clears the World Bank's High-income line on GNI per capita yet looks upper-middle on the other indicators? If you were deciding which countries may borrow on concessional terms, which of the two classifications would you use, and why?

*Your answer here:*

---

## AI-Assisted Expansion: choosing K by silhouette, your prompt, checked by you

Step 4 tried nine values of K and kept the one with the highest silhouette. A function that does
that for any table of features is something you would reuse in every clustering project, and it is
quick for AI to write. It also makes every choice your prompt leaves out, and on this data most of
those choices change the numbers. As in Labs 16 and 19, no prompt is given: you write it.

**The target.** Run on `df` with `feature_names`, for K from 2 to 10, your function must reproduce
Step 4: all nine silhouettes, from **0.2729 at K = 2** to **0.1792 at K = 10**, and the choice
**K = 2**. Asked for K from 3 to 10 only, it must choose **K = 4**, Step 3's K, with the same
silhouettes. (The check reads these numbers from your own Step 4, not from this text.)

**What the check calls.** `choose_k(data, columns, k_values)`: `data`, a DataFrame with one row
per country and the raw, unstandardized values; `columns`, a list of the column names to cluster
on; and `k_values`, a list of whole numbers. It gives back two things: the chosen K, then a pandas
DataFrame with one row per K and columns named `k` and `silhouette`. The check runs it on a copy
of `df`, which by now also holds the `Country` names, the `aggregate` and `income_level` columns,
the `_imputed` flags and the `cluster` and `wb_income` columns, with `feature_names`, for K = 2 to 10 and for K = 3 to 10. While it runs, the
check hides the lab's own variables (`df`, `X_scaled`, `scaler`, `sil_list`, ...), so the
function must work from its arguments alone.

**What a good prompt states.** Parts 2-4 and the paragraph above hold every answer.
- the function's name, its arguments in order and their types, and what it gives back, with the
  table's column names;
- the data: what one row is, that it arrives unstandardized, that it holds other columns too, and
  which ones to cluster on;
- the scaling: which scaler, and which rows it is fitted on;
- the values of K to try: where they come from;
- the K-Means settings that change the numbers: the initialization, `n_init` and `random_state`;
- the score: which one, and on which version of the data it is computed;
- how K is chosen from the scores;
- what to leave out (example data, downloads, charts), and a comment on each line.

**What to do**
1. Write your round 1 prompt in the cell below and send it, exactly as written, to Claude
   ([claude.northeastern.edu](https://claude.northeastern.edu)) or ChatGPT, in a new chat.
2. Read the reply before you run anything. Paste only the function into the round 1 cell, and run
   the check below it.
3. If a line says `NO`, find the choice in the function that caused it, state that choice in a
   round 2 prompt, and paste and check the new function in the round 2 cells. A third round goes
   over round 2's function. Stop when every line says `yes` and you can explain each line, or when
   you decide to stop.
4. If everything says `yes` in round 1, skip rounds 2 and 3. Instead, in Question 1, name a
   choice the AI made that your prompt never stated, and write the line you would add to your
   prompt to state it. In the log, write "not needed" in round 2's row.
5. Fill in the checking log and answer the two questions. Those and your prompts are graded, not
   the code.

**Reading the check.** For each range of K it prints the chosen K beside Step 4's, whether the
AI's table has the K values asked for, and then one line per K: Step 4's silhouette, the AI's,
`same?`, and the % apart. Your numbers and the AI's come from the same data in the same session,
so a correct function agrees in every printed digit, whether it gets the rows as a DataFrame or an
array. There is no allowed gap. Two numbers can look the same to four decimals and still say `NO`:
read the "% apart" column.

A `NO` means different from Step 4, not worse. With one start (`n_init='auto'`), the clusters
depend on where K-Means happened to start; an AI that writes `n_init=10` keeps the best of ten
starts, which is arguably the better setting, and gets different silhouettes. If that happens,
say in your log which setting you would keep, and why.

Keep the function's name and argument order. If the AI also wrote example data, downloaded the
data again or called its own function, leave that out: it can replace the lab's.

In [ ]:
# GUIDED — run as-is (you need not follow this code; the cells below use it)
# The check, as a function: the cells below run it once for each round.
# It saves the lab's data, model and Step 4's numbers now, before anything is pasted.
saved_df, saved_names = df.copy(), list(feature_names)
saved_X, saved_sil, saved_best = X_scaled.copy(), list(sil_list), best_k
saved_kmeans, saved_labels = kmeans, kmeans.labels_.copy()
saved_scaler, saved_mean = scaler, scaler.mean_.copy()
step4 = dict(zip(K_range, saved_sil))          # Step 4's silhouette for each K

# the lab's own variables, hidden while the AI's function runs
LAB_NAMES = ['df', 'feature_names', 'X_scaled', 'scaler', 'kmeans', 'km', 'labels', 'K_range',
             'sil_list', 'wcss_list', 'best_k', 'pca', 'X_pca', 'df_compare', 'cluster_means',
             'centroids', 'indicators']


def lab_data_unchanged():
    """True if df, X_scaled, kmeans, scaler and Step 4's numbers still hold what the lab made."""
    # try runs the lines under it; if they raise an error (say df is gone), except runs instead
    try:
        return bool(df.equals(saved_df) and feature_names == saved_names
                    and np.array_equal(X_scaled, saved_X)
                    and sil_list == saved_sil and best_k == saved_best
                    and kmeans is saved_kmeans and np.array_equal(kmeans.labels_, saved_labels)
                    and scaler is saved_scaler and np.array_equal(scaler.mean_, saved_mean))
    except Exception:
        return False


def show_pair(label, mine, ai):
    """One line: your number, the AI's, whether they agree, and how far apart they are."""
    # rtol=1e-9: equal to about 9 significant digits, so only rounding inside the computer passes
    same = "yes" if np.isclose(mine, ai, rtol=1e-9, atol=0) else "NO"
    pct = 100 * (ai - mine) / abs(mine)
    # a gap under 1%: sign and 2 significant digits, so a tiny one prints like +9.6e-06;
    # otherwise a whole number with its sign, like -20
    apart = f"{pct:+.2g}" if abs(pct) < 1 else f"{pct:+.0f}"
    print(f"  {label:<20}{mine:>10.4f}{ai:>10.4f}{same:>7}{apart:>11}%")


def describe(value):
    """What the function gave back, in a few words."""
    if isinstance(value, (tuple, list)):
        return f"a {type(value).__name__} of {len(value)}: " + ", ".join(type(v).__name__ for v in value)
    if isinstance(value, (int, float, np.number)):
        return f"one number, {value}"
    return f"a {type(value).__name__}"


def as_table(table):
    """The table with lower-case column names, k as a column and the score named silhouette, or None."""
    table = table.rename(columns=lambda c: str(c).lower())
    if "k" not in table.columns and str(table.index.name).lower() == "k":
        table = table.reset_index().rename(columns=lambda c: str(c).lower())
    score = [c for c in table.columns if c in ("silhouette", "silhouette_score")]
    if "k" not in table.columns or not score:
        return None
    return table.rename(columns={score[0]: "silhouette"})


def check_choose_k(fn):
    """Run the AI's choose_k on a copy of df, for K = 2-10 and K = 3-10, beside Step 4."""
    if not lab_data_unchanged():
        print("The pasted code changed df, X_scaled, kmeans, scaler or Step 4's numbers. "
              "Paste only the function, then rerun the notebook from the top.")
        return
    # callable(x) is True when x is a function that can be called
    if not callable(fn):
        print("choose_k should be a function. Paste the function the AI wrote.")
        return
    notebook = globals()          # every name defined so far in this notebook
    for k_values in (list(range(2, 11)), list(range(3, 11))):
        mine_k = max(k_values, key=lambda k: step4[k])      # Step 4's best K in this range
        hidden = {name: notebook.pop(name) for name in LAB_NAMES if name in notebook}
        try:
            # a copy of the data, so the function cannot change the lab's
            result = fn(saved_df.copy(), list(saved_names), list(k_values))
        except Exception as err:
            print(f"With K from {k_values[0]} to {k_values[-1]}, the AI's function stopped with an "
                  f"error. {type(err).__name__}: {err}")
            return
        finally:
            notebook.update(hidden)               # the lab's variables come back
        parts = list(result) if isinstance(result, (tuple, list)) else [result]
        ai_ks = [p for p in parts if isinstance(p, (int, np.integer)) and not isinstance(p, bool)]
        frames = [p for p in parts if isinstance(p, pd.DataFrame)]
        table = as_table(frames[0]) if frames else None

        print(f"\nK from {k_values[0]} to {k_values[-1]}")
        print(f"  {'':<20}{'Step 4':>10}{'AI':>10}{'same?':>7}{'% apart':>12}")
        if ai_ks:
            print(f"  {'chosen K':<20}{mine_k:>10}{int(ai_ks[0]):>10}"
                  f"{'yes' if ai_ks[0] == mine_k else 'NO':>7}")
        if frames and table is None:
            print(f"  choose_k gave back a table with the columns {list(frames[0].columns)}.")
        elif not ai_ks or table is None:
            print(f"  choose_k gave back {describe(result)}.")
        if not ai_ks or table is None:
            print("  The check needs two things: the chosen K, then a DataFrame with a column "
                  "named k and one named silhouette.")
            return
        try:
            table_ks = sorted(int(v) for v in table["k"])
        except Exception:
            print("  The table's k column should hold whole numbers.")
            return
        print(f"  K values in the AI's table: {table_ks}   "
              f"same as asked? {'yes' if table_ks == k_values else 'NO'}")
        for k in k_values:
            rows = table[table["k"] == k]
            if rows.empty:
                print(f"  {'silhouette, K = ' + str(k):<20}{step4[k]:>10.4f}{'missing':>10}")
                continue
            try:
                ai_value = float(rows["silhouette"].iloc[0])
            except Exception:
                print(f"  {'silhouette, K = ' + str(k):<20}{step4[k]:>10.4f}   not a number")
                continue
            show_pair(f"silhouette, K = {k}", step4[k], ai_value)
    print("\nIf a line says NO, find out why before you trust the function.")

**Your prompt, round 1.** Write it here, then send it exactly as written.

*Your prompt here:*

In [ ]:
# Paste the choose_k function from round 1 here, then run the check below.
# Paste only the function: if the AI also wrote example data or a call to it, leave that out.

In [ ]:
# CHECK — round 1
# To rerun this check, rerun the round 1 paste cell first: it checks whatever was pasted last
# globals() holds every name defined so far in this notebook
if "choose_k" not in globals():
    print("Paste the AI's choose_k function in the cell above first, keeping that name, "
          "and run that cell.")
else:
    choose_k_r1 = choose_k      # kept, so the round 2 check can tell the two apart
    check_choose_k(choose_k_r1)

**Your prompt, round 2** (and round 3 under it, if you need one). Keep what worked in round 1,
and state what the AI had to choose on its own.

*Your prompt here:*

In [ ]:
# Paste the choose_k function from round 2 here, then run the check below.
# Paste only the function: if the AI also wrote example data or a call to it, leave that out.

In [ ]:
# CHECK — round 2 (and round 3)
# "is" asks whether two names hold the very same function: if so, nothing new was pasted
if "choose_k" not in globals() or choose_k is globals().get("choose_k_r1"):
    print("Paste the AI's round 2 choose_k function in the cell above first, "
          "keeping that name, and run that cell.")
else:
    check_choose_k(choose_k)

### Checking log

This is graded, with your prompts and the two answers below. One row per round (a third round,
if any, goes in round 2's row): what you checked, how you checked it, and what you found. A row
that names the line that differed, how far apart it was, and the line of the AI's code
responsible earns credit; "it worked" does not.

| Round | What I checked | How | What I found |
|---|---|---|---|
| 1 |  |  |  |
| 2 (write "not needed" if round 1 was all yes) |  |  |  |

**In your own words.** Quote the lines of your final function that standardize the data, fit
K-Means and score it, and explain in 2-3 sentences what they do and why they give Step 4's
numbers.

*Your explanation here:*

### Questions

1. **Critique.** Name one choice the AI made that your prompt never stated, quote the line where it makes it, and say whether you would keep it and why. (A choice that caused a `NO` counts. If everything said `yes` in round 1, name such a choice and write the prompt line that would state it; then name a line that would give wrong numbers or break on other data: for example data with a blank in it, a column of text among `columns`, or K = 1, or a bare 4 instead of a list.)
2. **Judgment: where you stopped.** Which round did you stop at, and why do you, or do you not, trust that function to choose K on data the check has never seen? Then: when would you ask AI to write a loop like this, and when would you write it yourself?

*Your answers here:*

1.
2.

---

## Part 6: Peer Programming — Cluster Debate (Extension — 15 min)

Optional. Work with a partner. One person fits **K=3**, the other **K=5**, by changing `K_debate`
in the cell below. It draws your clusters on Step 3b's two principal components and prints each
cluster's size and three of its averages. Then debate:

1. Which value of K produces more **interpretable** clusters?
2. Can you name each cluster with an economically meaningful label?
3. Which cluster assignment would you use if advising the World Bank?

In [ ]:
# GUIDED — run as-is (optional); Partner B changes K_debate to 5
# Part 6: K-Means with K = 3 or 5 on the ten standardized indicators, drawn on Step 3b's two components
K_debate = 3

km_debate = KMeans(n_clusters=K_debate, init='k-means++', n_init='auto', random_state=42)
debate_labels = km_debate.fit_predict(X_scaled)

fig, ax = plt.subplots(figsize=(9, 6))
for k in range(K_debate):
    mask = debate_labels == k
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], label=f'Cluster {k}', alpha=0.7)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title(f'K-Means with K={K_debate}')
ax.legend()
plt.tight_layout()
plt.show()

print(f'K={K_debate} — Silhouette Score: {silhouette_score(X_scaled, debate_labels):.4f}\n')
for k in range(K_debate):
    mask = debate_labels == k
    print(f'Cluster {k}: {mask.sum()} countries | '
          f'GDP/cap ${df.loc[mask, "gdp_per_capita_ppp"].mean():,.0f} | '
          f'Life Exp {df.loc[mask, "life_expectancy"].mean():.1f} | '
          f'Infant Mort {df.loc[mask, "infant_mortality"].mean():.1f}')

**Expected output with K=3:** a silhouette of 0.1895 (Step 4's K=3 row), and clusters of 56, 88
and 65 countries. In the last lines, `:,.0f` prints a number with thousands commas and no decimals.

**Your answers** to the three debate questions:

*Your answers here:*

---

## Challenge: Cluster California Housing Block Groups (Take-Home)

There is no starter code: write it in the cell below and in new cells under it.

Apply the same pipeline to the **California Housing** dataset you have used since Lab 4: 20,640
block groups, loaded with `fetch_california_housing(as_frame=True)` from `sklearn.datasets`, whose
`.frame` is a DataFrame. Instead of countries by development indicators, cluster block groups by
housing and demographic features.

1. Drop the target, `MedHouseVal`: this is *unsupervised* learning.
2. Standardize the other eight features.
3. Find a K from 2 to 8 with the elbow and the silhouette. `silhouette_score` compares every pair
   of rows, so its cost grows with the *square* of the sample: on 20,640 rows one call takes
   minutes and looks like a hang. Score a random subset with
   `silhouette_score(X, labels, sample_size=2000, random_state=42)`. The elbow (`km.inertia_`)
   is cheap and needs no subsample.
4. Fit K-Means with your K, draw it on two principal components, and print each cluster's mean of
   every feature.

**Question:** Do California's block groups cluster into economically meaningful groups? Label each
cluster from its means (e.g., "urban high-income", "rural low-income", "coastal expensive"), and
say which features drive the split.

If you skip the Challenge, leave its cell unrun, so your submitted notebook shows no error.

In [ ]:
# YOUR TASK — take-home Challenge: write your code here, and in new cells below

*Your answer here:*

---
## Digital Portfolio: Your README

Copy the prompt below into Claude or ChatGPT. Fill in each [YOUR VALUE] from your own output
first, and do not quote a number this notebook did not print. **Do NOT ask the AI to write
Python code — only documentation.**

```
I need help writing a project README for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Downloaded 10 World Bank development indicators and kept
  [YOUR VALUE] countries; [YOUR VALUE]% of the Gini values had to be
  filled with the median
* Standardized the indicators and fitted K-Means with K=4: clusters
  of [YOUR VALUE] countries
* Compared K from 2 to 10: the silhouette was highest at K=[YOUR VALUE]
  ([YOUR VALUE]), against [YOUR VALUE] at K=4
* Drew the clusters on two principal components, which keep
  [YOUR VALUE]% of the variance
* Compared the clusters with the World Bank's income groups:
  [YOUR RESULT]
* Wrote my own prompt asking an AI for a function that chooses K by
  silhouette, went [YOUR VALUE] rounds, and checked it against my
  numbers: [YOUR RESULT]

**Please write a README.md entry including:**
1. Project Title: Clustering Economies with K-Means
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'robust', 'comprehensive'
or 'leverage'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ3916-lab22-clustering`

**First time?** Read the **GitHub Setup** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ3916-lab22-clustering`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ3916-lab22-clustering`, branch `main`,
   commit message `Lab 22 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.